In [ ]:
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler, RobustScaler

In [ ]:
INPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\splited"
OUTPUT_PATH = r"C:\Users\Gonzalo\Documents\GitHub\TesisAustral\Archivos_tesis\intermediate\normalized"
os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
X_train = pd.read_parquet(f"{INPUT_PATH}/X_train.parquet")
X_test = pd.read_parquet(f"{INPUT_PATH}/X_test.parquet")

original_n_columns = X_train.shape[1]
print(f"Columnas originales (antes de normalizar): {original_n_columns}")

scalers = {
    "Standard": StandardScaler(),
    "MinMax": MinMaxScaler(),
    "MaxAbs": MaxAbsScaler(),
    "Robust": RobustScaler(),
    "NoNorm": None
}

📊 Columnas originales (antes de normalizar): 539
✔ Standard: columnas después de normalizar = 539
✔ MinMax: columnas después de normalizar = 539
✔ MaxAbs: columnas después de normalizar = 539
✔ Robust: columnas después de normalizar = 539
✔ NoNorm: columnas después de normalizar = 539

📝 Resumen guardado en: resumen_columnas_normalizacion.csv


In [ ]:
#Guardar métricas
metricas = []

for name, scaler in scalers.items():
    norm_dir = os.path.join(OUTPUT_PATH, name)
    os.makedirs(norm_dir, exist_ok=True)

    if scaler:
        scaler.fit(X_train)
        X_train_norm = pd.DataFrame(scaler.transform(X_train), columns=X_train.columns, index=X_train.index)
        X_test_norm = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)
    else:
        X_train_norm = X_train.copy()
        X_test_norm = X_test.copy()

    X_train_norm.to_parquet(f"{norm_dir}/X_train_{name}.parquet")
    X_test_norm.to_parquet(f"{norm_dir}/X_test_{name}.parquet")
    
    y_train = pd.read_parquet(f"{INPUT_PATH}/y_train.parquet")
    y_test = pd.read_parquet(f"{INPUT_PATH}/y_test.parquet")
    y_train.to_parquet(f"{norm_dir}/y_train_{name}.parquet")
    y_test.to_parquet(f"{norm_dir}/y_test_{name}.parquet")

    n_cols_train = X_train_norm.shape[1]
    print(f"{name}: columnas después de normalizar = {n_cols_train}")

    # Guardar en resumen
    metricas.append({
        "normalizacion": name,
        "columnas_originales": original_n_columns,
        "columnas_resultantes": n_cols_train
    })

# Guardar resumen en CSV
df_metricas = pd.DataFrame(metricas)
df_metricas.to_csv(os.path.join(OUTPUT_PATH, "resumen_columnas_normalizacion.csv"), index=False)
print("\nResumen guardado en: resumen_columnas_normalizacion.csv")